<a href="https://colab.research.google.com/github/rushitpatel2311/Group_7_Deepfake_Detection/blob/rushit/Group_7_Deepfake_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group 7 — AI-Driven Deepfake Detection System

**Module:** AI Systems Engineering (CMP-L044) — Part 2 Artefact

---

## Notebook Structure

| Section | Lead | Description |
|---------|------|-------------|
| **Section 0** | Rushitkumar Patel (A00085504) | Install packages, dataset setup, global config |
| **Section 1** | Isha Luhar (A00085061) | Data ingestion, MTCNN preprocessing, Mel spectrograms |
| **Section 2** | Nishtha Solanki (A00087199) | Video pipeline — EfficientNet-B4 + Transformer |
| **Section 3** | OM Mistry (A00067376) | Audio pipeline — ResNet-18 on Mel spectrograms |
| **Section 4** | Rushitkumar Patel (A00085504) | Late fusion meta-learner + Grad-CAM explainability |
| **Section 5** | All | Evaluation, robustness, fairness, MLflow logging |

---

## ⚠️ Before You Run

1. **Runtime → Change runtime type → T4 GPU** (required)
2. **Add Kaggle .json file of Legacy API Credentials to 0.2 of Environment setup section**
3. Run cells **strictly top to bottom** — all sections are interdependent
4. If Colab disconnects, re-run from Cell 0.1 (packages reset each session)

---


# SECTION 0 — Environment Setup
**Rushitkumar Patel (A00085504)** \
Run every time Colab restarts.

In [1]:
#================================ Install dependencies ================================
import sys

print("Installing packages...")
!pip install -q \
    timm \
    facenet-pytorch \
    librosa \
    mlflow \
    scikit-learn \
    opencv-python-headless \
    albumentations \
    matplotlib \
    seaborn \
    tqdm \
    pandas \
    Pillow \
    scipy

print("✅ All packages installed.")

Installing packages...
✅ All packages installed.


In [2]:
#================================ Dataset Stup ================================
""" here we use kaggle dataset for video and audio.
    for that we upload kaggle.json (download from kaggle account from Legacy API Credentials) file to colab
    for video dataset we used alternative of DFDC dataset (ucimachinelearning/deep-fake-detection-cropped-dataset)
    for audio dataset we used alternative of ASVspoof 2021 (adarshsingh0903/audio-deepfake-detection-dataset)

    this datasets are smaller compair to original because we used free version of colab which limited the access to virtual memory and GPU
"""

import os
import zipfile
from google.colab import files

# 1. Authenticate (Upload kaggle.json)
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Please upload your kaggle.json file (Legacy API Key):")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# 2. Define Public IDs (Avoids 403 Errors and Disk Overflow)
VIDEO_ID = "ucimachinelearning/deep-fake-detection-cropped-dataset"
AUDIO_ID = "adarshsingh0903/audio-deepfake-detection-dataset"

# 3. Create structure for EfficientNet-B4 and ResNet-18
DATA_ROOT = '/content/data'
for folder in ['audio/real', 'audio/fake', 'train/real', 'train/fake']:
    os.makedirs(os.path.join(DATA_ROOT, folder), exist_ok=True)

# 4. Download Video Dataset
print("📥 Downloading Video Dataset (Public)...")
!kaggle datasets download -d {VIDEO_ID} -p /content/video_temp --unzip

# 5. Download Audio Dataset (Smaller 1GB version instead of 58GB)
print("📥 Downloading Audio Dataset (Optimized Size)...")
!kaggle datasets download -d {AUDIO_ID} -p /content/audio_temp --unzip

# 6. Organize Files
print("📂 Organizing files...")

# Move Audio to match your AudioDataset class requirements
# Check internal folder names: if 'real' and 'fake' are nested, move them
!mv /content/audio_temp/audio_deepfake/real/* /content/data/audio/real/ 2>/dev/null
!mv /content/audio_temp/audio_deepfake/fake/* /content/data/audio/fake/ 2>/dev/null

# Move Video to match your DeepfakeFrameDataset class requirements
!mv /content/video_temp/*.mp4 /content/data/train/fake/ 2>/dev/null

print(f"✅ Setup Complete. DATA_ROOT: {DATA_ROOT}")

Please upload your kaggle.json file (Legacy API Key):


Saving kaggle.json to kaggle.json
📥 Downloading Video Dataset (Public)...
Dataset URL: https://www.kaggle.com/datasets/ucimachinelearning/deep-fake-detection-cropped-dataset
License(s): CC0-1.0
100% 666M/666M [00:33<00:00, 21.1MB/s]

📥 Downloading Audio Dataset (Optimized Size)...
Dataset URL: https://www.kaggle.com/datasets/adarshsingh0903/audio-deepfake-detection-dataset
License(s): apache-2.0
100% 1.09G/1.09G [01:04<00:00, 18.1MB/s]

📂 Organizing files...
✅ Setup Complete. DATA_ROOT: /content/data


In [3]:
#================================ Global imports & configuration ================================
import os, random, warnings, time
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.models as models

import timm
import librosa
import librosa.display
import cv2
import mlflow
import mlflow.pytorch

import albumentations as A
from albumentations.pytorch import ToTensorV2
from facenet_pytorch import MTCNN

from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings('ignore')

#================================ Reproducibility ================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

#================================ Device ================================

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   ⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

# Define DEMO_MODE here since the Kaggle cell does not set it.
# Assuming successful Kaggle download means not in demo mode.
DEMO_MODE = False

#================================ Global config ================================

CFG = {
    # Data
    'data_root'      : DATA_ROOT,
    'img_size'       : 224,
    'sample_rate'    : 16000,   # Hz
    'n_mels'         : 128,     # mel bins
    'hop_length'     : 160,     # 10 ms @ 16 kHz
    'win_length'     : 400,     # 25 ms window
    'n_fft'          : 512,
    'audio_duration' : 3.0,     # seconds per clip
    'spec_width'     : 300,     # time frames (fixed)
    # Training
    'batch_size'     : 16,
    'epochs_video'   : 5,
    'epochs_audio'   : 5,
    'lr'             : 1e-4,
    'weight_decay'   : 1e-4,
    'label_smoothing': 0.1,
    # MLflow
    'experiment'     : 'Group7_Deepfake_Detection',
    # Part 1 NFR targets
    'latency_budget_ms' : 500,
    'auc_target'        : 0.92,
    'fpr_target'        : 0.03,
    'fairness_target'   : 0.05,
}

#================================ MLflow experiment ================================

mlflow.set_experiment(CFG['experiment'])

print(f"\n📊 MLflow experiment : {CFG['experiment']}")
print(f"   DEMO_MODE         : {DEMO_MODE}")
print("\n✅ Configuration complete.")

🖥️  Device: cpu
   ⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU


2026/05/03 11:37:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/03 11:37:46 INFO mlflow.store.db.utils: Updating database tables
2026/05/03 11:37:49 INFO mlflow.tracking.fluent: Experiment with name 'Group7_Deepfake_Detection' does not exist. Creating a new experiment.



📊 MLflow experiment : Group7_Deepfake_Detection
   DEMO_MODE         : False

✅ Configuration complete.
